# ACC long-row shortener, shared-record rows (job J6, Mad Hatters, 6 Oct)

**What it does:** takes our roads on the 44 rows where 2 to 6 teams share the record and we are within 30 moves (closest first). A shorter road here beats the record outright. Run it in a second tab next to ACC_LONGROWS_COLAB_v2; it writes to its own folder. It shortens the roads with the exact window optimizer: each road is cut into pieces that run in parallel and are stitched back together, and every result is checked on the pinned official verifier.

**No model, no SAIR calls, no repo pull.** The kit is about 25 KB; the official verifier (23 MB, public) is cloned once per session.

**Steps:** 1) connect Drive, 2) unpack the kit, 3) run. **Keep this tab open.** Free Colab disconnects idle sessions.

Every improvement is saved at once to the `COLAB_LONGROWS_K2` folder in the hub. If Colab disconnects, just **run all again**: it resumes from `state.jsonl` and skips rows it already finished.

Shorter roads go to `COLAB_LONGROWS_K2/LINES_COLAB_LONGROWS.txt`, and UPG-H's check-in picks them up and checks them against the board for Jeremy to submit.

In [ ]:
#@title 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Unpack the kit from the hub
import zipfile, pathlib
HUB = pathlib.Path('/content/drive/MyDrive/Mad Hatters ACC Shared Hub')
zips = sorted(HUB.glob('ACC_LONGROWS_KIT_*.zip')) or sorted(pathlib.Path('/content/drive/MyDrive').rglob('ACC_LONGROWS_KIT_*.zip'))
assert zips, 'Put ACC_LONGROWS_KIT_*.zip in the Mad Hatters ACC Shared Hub folder'
KIT = pathlib.Path('/content/acc_kit')
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(KIT)
print('kit:', zips[-1].name, sorted(p.name for p in KIT.iterdir()))

In [ ]:
#@title 3. Run (about 11 hours; safe to stop and rerun, it resumes)
hours = 11 #@param {type:"number"}
rows = '' #@param {type:"string"}
import subprocess, sys, re
OUT = HUB / 'COLAB_LONGROWS_K2'
DEFAULT = 'ac-00631,ac-00956,ac-04642,ac-05805,ac-06468,ac-02347,ac-02591,ac-08367,ac-01205,ac-01932,ac-07661,ac-07540,ac-02033,ac-09416,ac-04826,ac-01061,ac-08847,ac-00058,ac-00600,ac-03079,ac-06796,ac-06084,ac-02388,ac-02900,ac-06834,ac-07419,ac-00106,ac-06727,ac-07028,ac-07184,ac-03828,ac-05586,ac-06899,ac-01459,ac-01715,ac-03602,ac-07663,ac-08656,ac-09751,ac-05032,ac-05505,ac-02662,ac-08014,ac-02873'
todo = [r.strip() for r in (rows or DEFAULT).split(',') if r.strip()]
# a row counts as finished when the log moved on to another row after its 'done at' line
done, last = set(), None
log = OUT / 'log.txt'
for line in (log.read_text().splitlines() if log.exists() else []):
    m = re.search(r'(ac-\d{5}): (start|done at)', line)
    if m and m.group(2) == 'start':
        if last: done.add(last)
        last = None
    elif m:
        last = m.group(1)
todo = [r for r in todo if r not in done]
print(len(done), 'rows already finished;', len(todo), 'to go')
cmd = [sys.executable, str(KIT / 'run_longrows.py'), '--out', str(OUT), '--hours', str(hours), '--piece', '48', '--rows', ','.join(todo)]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
print('exit', p.wait())